# Streaming structured RAG

In [2]:
!uv add jaxn

Resolved 122 packages in 796ms                                       
⠙ Preparing packages... (0/1)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/13.08 KiB           
⠙ Preparing packages... (0/1)---------- 13.08 KiB/13.08 KiB         
Prepared 1 package in 151ms                                                       
Installed 1 package in 12ms                                 
 + jaxn==0.0.5


In [3]:
from openai import OpenAI
from jaxn import JSONParserHandler, StreamingJSONParser
from pydantic import BaseModel, Field
from typing import List, Optional

In [4]:
openai_client = OpenAI()

In [6]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

stream = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    stream=True
)

response = None
for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response


Once upon a time, in a gloomy forest far away, there lived a unicorn named Umbra. Unlike most unicorns, who were vibrant and full of joy, Umbra had a dark coat that blended with the shadows and a mane that sparkled like stars in a night sky—beautiful yet haunting.

Umbra had dreams of making friends with the other creatures in the forest, but they all avoided him. The rabbits thumped their feet and scurried away, the birds perched high in the trees, screeching warnings that the dark unicorn was coming. Even the wise old owl would blink at Umbra, as if unsure whether he was a friend or foe.

One dreary day, Umbra sought to change his fate. He wandered deep into the forest, determined to find a way to light up his dreary existence. After hours of searching, he stumbled upon an ancient, twisted tree with glowing fruit hanging from its branches. Remembering tales of how the fruit could grant wishes, Umbra hesitated. He thought about how much he wanted to be accepted and loved.

With a powe

In [7]:
from pydantic import BaseModel, Field

class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

In [8]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

with openai_client.responses.stream(
    model="gpt-4o-mini",
    input=messages,
    text_format=ArticleResponse,
) as stream:
    response = None
    for event in stream:
        if hasattr(event, 'delta'):
            print(event.delta, end='')
        if hasattr(event, 'response'):
            response = event.response


{"title":"The Unicorn's Bad Day","subtitle":"A Tale of Misfortune and Mistakes","sections":[{"header":"Once Upon a Time","text":"In a mystical forest where the trees whispered secrets and the rivers sparkled like diamonds, lived a unicorn named Glitter. Unlike other unicorns, Glitter had a shimmering golden mane but was notoriously clumsy. One sunny morning, Glitter decided it was time for an adventure."},{"header":"The Great Fall","text":"Excited to explore the enchanting meadows, Glitter pranced through the forest, her hooves tapping a happy rhythm. Suddenly, she spotted a beautiful butterfly fluttering by. Mesmerized, Glitter leaped to chase it, forgetting she was beside a steep cliff. With a misstep, she tumbled down, landing in a puddle of muddy water."},{"header":"The Muddy Mishap","text":"Covered in mud and smelling like a swamp, Glitter stood up, embarrassed but determined to continue her adventure. She shook off the mud, but it only splattered everywhere, making her look even 

In [9]:
story = response.output_parsed

print('# ' + story.title)
print(story.subtitle)
print()

for section in story.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()


# The Unicorn's Bad Day
A Tale of Misfortune and Mistakes

## Once Upon a Time

In a mystical forest where the trees whispered secrets and the rivers sparkled like diamonds, lived a unicorn named Glitter. Unlike other unicorns, Glitter had a shimmering golden mane but was notoriously clumsy. One sunny morning, Glitter decided it was time for an adventure.


## The Great Fall

Excited to explore the enchanting meadows, Glitter pranced through the forest, her hooves tapping a happy rhythm. Suddenly, she spotted a beautiful butterfly fluttering by. Mesmerized, Glitter leaped to chase it, forgetting she was beside a steep cliff. With a misstep, she tumbled down, landing in a puddle of muddy water.


## The Muddy Mishap

Covered in mud and smelling like a swamp, Glitter stood up, embarrassed but determined to continue her adventure. She shook off the mud, but it only splattered everywhere, making her look even more ridiculous. The animals of the forest couldn't help but giggle at her misfor

## Parse JSON on the go

In [10]:
raw_json = """
{"title":"The Misadventures of Sparkle","subtitle":"A Tale of Trouble","sections":[{"header":"Once Upon a Time","text":"In a magical land far away, there lived a young unicorn named Sparkle."},{"header":"The Problem","text":"Sparkle had a mischievous streak that often led to chaos."}]}
""".strip()


In [12]:
from typing import Any

class ArticleResponseHandler(JSONParserHandler):

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)


In [13]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)


In [14]:
for c in raw_json:
    parser.parse_incremental(c)


# The Misadventures of Sparkle
A Tale of Trouble


## Once Upon a Time

In a magical land far away, there lived a young unicorn named Sparkle.

## The Problem

Sparkle had a mischievous streak that often led to chaos.

In [15]:
def llm_structured_stream(
    user_prompt,
    output_type,
    parser_handler=JSONParserHandler(),
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with openai_client.responses.stream(
        model=model,
        input=messages,
        text_format=output_type,
    ) as stream:
        response = None
        for event in stream:
            if hasattr(event, 'delta'):
                parser.parse_incremental(event.delta)
            if hasattr(event, 'response'):
                response = event.response

    return response


In [16]:
instructions = "your task is to tell the user bad time stories"
user_prompt = "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)


# The Tale of the Cursed Unicorn
A Dark Legend from the Enchanted Forest


## The Whispers of the Forest

In a magical realm, there lived a beautiful unicorn named Lumina. Unlike her kin, who brought joy and light to the world, Lumina possessed a dark secret. A sorceress, jealous of her beauty, cursed Lumina, turning her radiant horn into a twisted, jagged spike. As the unicorn wandered through the enchanted forest, her sad cries echoed among the trees, echoing like silent screams.

## The Warning

Folk in nearby villages spoke of the cursed unicorn in hushed tones. They warned travelers not to enter the forest at dusk, for Lumina’s presence brought despair. Those who sought her glittering mane often found themselves lost, trapped in swirling mists, forever wandering, haunted by the sadness she radiated.

## A Brave Adventurer

One fateful evening, a brave but foolish adventurer named Theo, driven by tales of magic, decided to seek out Lumina. Ignoring the warnings, he ventured into th

## Streaming Documentation RAG

In [17]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )


Indexed 385 chunks from 95 documents


In [18]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")


In [19]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)


In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

In [21]:
response = rag('llm as a judge')

In this context, evaluating a text using a Large Language Model (LLM) as a judge involves two primary approaches:

1. **Reference-based Evaluation**: This method compares new responses against approved or "ground truth" responses. It is particularly useful for regression testing when you have established correct responses to compare against.

2. **Open-ended Evaluation**: This approach allows for assessment based on custom criteria when no reference output exists. It helps to evaluate the quality of new outputs that may vary widely in nature.

The tutorial highlights how to create and tune the LLM evaluator using these methods, focusing on setting up your evaluation dataset, running the LLM as a judge, and evaluating the effectiveness of the judge's evaluations against manual labels. You will also learn to develop prompts that capture specific evaluation criteria, such as correctness and verbosity.

For implementation, a basic understanding of Python and an OpenAI API key are necessary